# 📊 Code 3 V5: Comprehensive Feature Engineering

## What's new in V5 (bug fixes from V4 audit)
1. **MaxDrawdown single-pass** — V4 used a double rolling window, delaying the first valid value to day 2×window (e.g. `MaxDrawdown_252d` started ~July 2011 instead of ~July 2010). This also fixes `Calmar_*` which divides by it.
2. **Gap-free market series (Step 4b)** — one canonical Nifty 500 return series, aligned to every stock date with the price forward-filled. Fixes the 2014 all-stock `Beta`/`R²`/`Idio_Vol` NaN block and any single-day holes in `Excess_Ret_N500_*` and market-regime features.
3. **min_periods on Beta, Sharpe, Sortino** — a single mid-window NaN can no longer blank a whole rolling window.
4. **Validation cell (Step 4c)** — asserts the market series has no interior NaN gaps before any market-dependent feature is computed.
5. **First-valid-date report (Step 30b)** — per-feature first non-NaN row, to catch window-start bugs at a glance.

## Purpose
Build a comprehensive feature set (~166 features) for both **XGBoost** and **LSTM** models.

## Inputs (upload to Colab)
1. **`data_clean.csv`** (from Code 2) — Stock OHLCV + Sector/Industry
2. **`indices_all.csv`** (from Code 1c) — For Nifty 500 returns & market regime
3. **`stock_index_mapping.csv`** (from Code 1e) — For market cap & index classification

## Outputs
1. **`data_features.csv`** — Main dataset with ~166 features (~1.5 GB)
2. **`feature_dictionary.csv`** — Documentation for every feature (Name, Category, Formula, LSTM/XGB use)
3. **`feature_health_report.csv`** — NaN %, distributions, basic stats
4. **`feature_first_valid_report.csv`** — First non-NaN row per feature (window-start sanity check)

## Feature Categories
| Category | Count |
|----------|-------|
| Base data (OHLCV, meta) | 7 |
| Multi-horizon Returns (1d, 2d, 3d, 5d, 10d, 20d, 60d, 120d, 252d) | 9 |
| Multi-horizon Volatility (5d to 252d) | 6 |
| Risk-adjusted Returns (Sharpe, Sortino, Calmar) | 16 |
| Drawdown features (MaxDD, DaysSincePeak) | 12 |
| Up-days percentage | 6 |
| Returns distribution (Skew, Kurtosis) | 8 |
| Multi-horizon Volume ratios | 6 |
| Technical Indicators (RSI, MACD, BB, ATR, MFI, ADX) | 12 |
| Moving Averages & Price ratios | 7 |
| 52-week features | 4 |
| Higher Highs / Lower Lows | 12 |
| Volume Indicators (OBV, CMF, Value Traded) | 5 |
| Daily price action | 4 |
| Excess Returns vs Nifty 500 | 9 |
| Excess Returns vs Sector | 9 |
| Beta, R², Idiosyncratic Vol (60d & 252d) | 6 |
| Cross-sectional Ranks | 11 |
| Market Regime | 5 |
| Market Cap | 4 |
| Sector (raw + encoded) | 3 |
| Date Features (raw + cyclical) | 11 |
| **TOTAL** | **~172** |

## ⚠️ Important Notes
- **Runtime:** ~35-50 minutes on Colab free
- **Memory:** Will use ~6-9 GB (float32 optimization applied)
- **First 252 days NaN:** Long-horizon features require history. Effective data starts ~Jan 2012.
- **No leakage:** All features use only past data (≤ current date)

---

## Step 1: Install Required Packages

In [1]:
# Install technical analysis library
!pip install ta --quiet

print("✅ Packages installed!")

  Preparing metadata (setup.py) ... done
✅ Packages installed!


## Step 2: Import Libraries & Configuration

In [2]:
import pandas as pd
import numpy as np
import ta
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
import time
import gc
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy: {np.__version__}")
print(f"   Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Track overall runtime
overall_start = time.time()

✅ Libraries imported!
   Pandas: 2.2.2
   NumPy: 2.0.2
   Run started: 2026-06-16 21:01:29


In [3]:
# Configuration
INPUT_FILE_DATA = 'data_clean.csv'
INPUT_FILE_INDICES = 'indices_all.csv'
INPUT_FILE_MAPPING = 'stock_index_mapping.csv'

OUTPUT_FILE_FEATURES = 'data_features.parquet'
OUTPUT_FILE_DICT = 'feature_dictionary.csv'
OUTPUT_FILE_HEALTH = 'feature_health_report.csv'

# Multi-horizon configuration
RETURN_HORIZONS = [1, 2, 3, 5, 10, 20, 60, 120, 252]
VOL_HORIZONS = [5, 10, 20, 60, 120, 252]  # Skipping 1-3d (mathematically noisy)
DISTRIBUTION_HORIZONS = [20, 60, 120, 252]  # Need ≥20 points for skew/kurt
HHLL_HORIZONS = [10, 20, 60, 120]  # For higher highs/lower lows
BETA_HORIZONS = [60, 252]  # 60d recent + 252d stable

# Technical indicator periods
RSI_PERIOD = 14
ATR_PERIOD = 14
BB_PERIOD = 20
BB_STD = 2
ADX_PERIOD = 14
MFI_PERIOD = 14
CMF_PERIOD = 20

# Store feature metadata for dictionary
feature_metadata = []

print("✅ Configuration set!")
print(f"   Return horizons: {RETURN_HORIZONS}")
print(f"   Volatility horizons: {VOL_HORIZONS}")
print(f"   Distribution horizons: {DISTRIBUTION_HORIZONS}")
print(f"   Beta horizons: {BETA_HORIZONS}")

✅ Configuration set!
   Return horizons: [1, 2, 3, 5, 10, 20, 60, 120, 252]
   Volatility horizons: [5, 10, 20, 60, 120, 252]
   Distribution horizons: [20, 60, 120, 252]
   Beta horizons: [60, 252]


## Step 3: Load Input Data

In [4]:
print("Loading input files...")
print("-"*80)

# Load main stock data
print(f"📂 Loading {INPUT_FILE_DATA}...")
df = pd.read_csv(INPUT_FILE_DATA)
df['Date'] = pd.to_datetime(df['Date'])
print(f"   ✅ Loaded: {len(df):,} rows × {len(df.columns)} cols")
print(f"   Unique stocks: {df['Ticker'].nunique()}")
print(f"   Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"   Columns: {df.columns.tolist()}")
print()

# Load indices data
print(f"📂 Loading {INPUT_FILE_INDICES}...")
df_indices = pd.read_csv(INPUT_FILE_INDICES)
df_indices['Date'] = pd.to_datetime(df_indices['Date'])
print(f"   ✅ Loaded: {len(df_indices):,} rows × {len(df_indices.columns)} cols")
print(f"   Columns: {df_indices.columns.tolist()}")
print()

# Load stock-index mapping
print(f"📂 Loading {INPUT_FILE_MAPPING}...")
df_mapping = pd.read_csv(INPUT_FILE_MAPPING)
print(f"   ✅ Loaded: {len(df_mapping):,} rows × {len(df_mapping.columns)} cols")
print(f"   Columns: {df_mapping.columns.tolist()}")

print("-"*80)

Loading input files...
--------------------------------------------------------------------------------
📂 Loading data_clean.csv...
   ✅ Loaded: 2,417,660 rows × 10 cols
   Unique stocks: 570
   Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00
   Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker', 'Value_Traded', 'Value_Traded_Cr', 'Daily_Return']

📂 Loading indices_all.csv...
   ✅ Loaded: 4,787 rows × 16 cols
   Columns: ['Date', 'NIFTY_50', 'NIFTY_100', 'NIFTY_200', 'NIFTY_500', 'NIFTY_MIDCAP_100', 'NIFTY_BANK', 'NIFTY_IT', 'NIFTY_PHARMA', 'NIFTY_AUTO', 'NIFTY_FMCG', 'NIFTY_METAL', 'NIFTY_ENERGY', 'NIFTY_FINANCIAL_SERVICES', 'NIFTY_REALTY', 'NIFTY_MEDIA']

📂 Loading stock_index_mapping.csv...
   ✅ Loaded: 575 rows × 12 cols
   Columns: ['Symbol', 'Company_Name', 'Index_Classification', 'Classification_Date', 'Mapping_Status', 'Suggested_Index', 'Market_Cap_INR_Cr', 'Rank', 'Sector', 'Exchange', 'Ticker_Used', 'Reason']
-----------------------------------------

## Step 4: Initial Data Inspection & Sorting

Sort by Ticker and Date — **CRITICAL for time-series operations.**

In [5]:
print("Initial data inspection...")
print("-"*80)

# Sort by Ticker and Date (REQUIRED for time-series operations)
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
print("✅ Sorted by Ticker, Date")

# Check for required columns
required_cols = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print(f"❌ MISSING COLUMNS: {missing}")
    raise ValueError(f"Required columns missing: {missing}")
else:
    print("✅ All required columns present")

# Convert numeric columns to float32 for memory efficiency
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
for col in numeric_cols:
    df[col] = df[col].astype('float32')
print("✅ Converted numeric columns to float32 (memory optimization)")

# Memory usage
memory_mb = df.memory_usage(deep=True).sum() / (1024**2)
print(f"\n💾 Current memory: {memory_mb:.1f} MB")

# Initial stats
print(f"\n📊 Initial Statistics:")
print(f"   Total rows: {len(df):,}")
print(f"   Unique stocks: {df['Ticker'].nunique()}")
print(f"   Initial columns: {len(df.columns)}")

print("-"*80)

Initial data inspection...
--------------------------------------------------------------------------------
✅ Sorted by Ticker, Date
✅ All required columns present
✅ Converted numeric columns to float32 (memory optimization)

💾 Current memory: 249.4 MB

📊 Initial Statistics:
   Total rows: 2,417,660
   Unique stocks: 570
   Initial columns: 10
--------------------------------------------------------------------------------


## Step 4b: Build Gap-Free Market Return Series (FIX V5)

**Why this exists:** Several features (Excess Returns vs Nifty 500, Beta/R²/Idio Vol, Market Regime) depend on the Nifty 500 daily return series. In V4 these were merged with `how='left'` directly, so any date present in stock data but missing from the index produced a NaN — and when a rolling window sat on top (Beta), one NaN day cascaded into a full window of NaN for **all** stocks (the 2014 Beta block).

**Fix:** Build ONE canonical market return series here, aligned to every stock trading date, with the index *price* forward-filled before computing returns. A forward-filled price yields a 0% return on a genuine gap day and resumes correctly the next day — no NaN holes. All downstream market-dependent steps reuse this `market_returns_map`.

We also keep a multi-horizon version for the excess-return features.

In [6]:
print("Step 4b: Build gap-free market return series")
print("-"*80)

# Validate NIFTY_500 exists
if 'NIFTY_500' not in df_indices.columns:
    print("\u274c NIFTY_500 column not in indices file!")
    print(f"   Available: {df_indices.columns.tolist()}")
    raise ValueError("NIFTY_500 missing")

# All unique stock trading dates (the grid we must cover)
all_stock_dates = pd.DataFrame({'Date': sorted(df['Date'].unique())})
print(f"   Stock trading dates: {len(all_stock_dates):,}")

# Align index to stock dates, forward-fill the PRICE (not the return)
df_mkt = df_indices[['Date', 'NIFTY_500']].copy().sort_values('Date')
df_mkt = all_stock_dates.merge(df_mkt, on='Date', how='left')
n_missing_before = df_mkt['NIFTY_500'].isna().sum()
df_mkt['NIFTY_500'] = df_mkt['NIFTY_500'].ffill()
# Edge case: leading NaNs (stock dates before index history starts) -> back-fill
df_mkt['NIFTY_500'] = df_mkt['NIFTY_500'].bfill()
n_missing_after = df_mkt['NIFTY_500'].isna().sum()
print(f"   Index price gaps on stock dates: {n_missing_before} \u2192 filled, {n_missing_after} remaining")

# Canonical daily market return (gap-free)
# Build multi-horizon market returns from the gap-free price series.
# NOTE: do NOT add 'Market_Return_1d' separately — h=1 is in RETURN_HORIZONS,
# so the loop already creates it.  Adding it twice caused duplicate columns.
for h in RETURN_HORIZONS:
    df_mkt[f'Market_Return_{h}d'] = df_mkt['NIFTY_500'].pct_change(h) * 100

# Keep a clean lookup frame (Date -> market returns)
# 'Market_Return_1d' is included via h=1 in RETURN_HORIZONS (no duplicates)
market_cols_keep = ['Date'] + [f'Market_Return_{h}d' for h in RETURN_HORIZONS]
df_market_returns = df_mkt[market_cols_keep].copy()

print(f"\n\u2705 Gap-free market series built")
print(f"   Market_Return_1d NaN count (excl. first row): {df_market_returns['Market_Return_1d'].isna().sum()}")
print(f"   (1 expected NaN at the very first date from pct_change)")
print("-"*80)

Step 4b: Build gap-free market return series
--------------------------------------------------------------------------------
   Stock trading dates: 4,799
   Index price gaps on stock dates: 12 → filled, 0 remaining

✅ Gap-free market series built
   Market_Return_1d NaN count (excl. first row): 1
   (1 expected NaN at the very first date from pct_change)
--------------------------------------------------------------------------------


## Step 4c: Validate Market Series (FIX V5)

Sanity check that the gap-free market series is actually gap-free on the stock-date grid. If this fails, the Beta/Excess-return/Regime features would inherit NaN holes — better to catch it here than debug a 2014 Beta block later.

In [7]:
print("Step 4c: Validate market series")
print("-"*80)

# Use int() throughout — .isna().sum() returns a scalar Series when there
# are no duplicate columns, but int() makes the comparison unambiguous.
n_nan_1d = int(df_market_returns['Market_Return_1d'].isna().sum())
print(f"   Market_Return_1d NaN count: {n_nan_1d} (expected: 1)")

# Check there are no INTERIOR gaps (NaN after the first valid value)
ret_series = df_market_returns['Market_Return_1d']
first_valid_pos = int(pd.Series(ret_series.values).first_valid_index())
interior = ret_series.iloc[first_valid_pos + 1:]
interior_nan = int(interior.isna().sum())  # int() ensures scalar comparison

if interior_nan == 0:
    print("   \u2705 PASS: No interior NaN gaps in market return series")
else:
    print(f"   \u274c FAIL: {interior_nan} interior NaN gaps found!")
    bad_idx = interior[interior.isna()].index.tolist()
    bad_dates = df_market_returns.loc[bad_idx, 'Date'].tolist()
    print(f"   Problem dates (first 10): {bad_dates[:10]}")
    raise ValueError("Market return series has interior gaps - Beta/Excess features would be corrupted")

# Check multi-horizon market returns
for h in RETURN_HORIZONS:
    col = f'Market_Return_{h}d'
    if col not in df_market_returns.columns:
        print(f"   \u26a0\ufe0f  {col} missing from df_market_returns")
        continue
    s = df_market_returns[col]
    fv = int(pd.Series(s.values).first_valid_index())
    interior_h = int(s.iloc[fv + 1:].isna().sum())
    status = '\u2705' if interior_h == 0 else '\u26a0\ufe0f '
    print(f"   {status} {col}: first valid at row {fv}, interior gaps: {interior_h}")

print("-"*80)


Step 4c: Validate market series
--------------------------------------------------------------------------------
   Market_Return_1d NaN count: 1 (expected: 1)
   ✅ PASS: No interior NaN gaps in market return series
   ✅ Market_Return_1d: first valid at row 1, interior gaps: 0
   ✅ Market_Return_2d: first valid at row 2, interior gaps: 0
   ✅ Market_Return_3d: first valid at row 3, interior gaps: 0
   ✅ Market_Return_5d: first valid at row 5, interior gaps: 0
   ✅ Market_Return_10d: first valid at row 10, interior gaps: 0
   ✅ Market_Return_20d: first valid at row 20, interior gaps: 0
   ✅ Market_Return_60d: first valid at row 60, interior gaps: 0
   ✅ Market_Return_120d: first valid at row 120, interior gaps: 0
   ✅ Market_Return_252d: first valid at row 252, interior gaps: 0
--------------------------------------------------------------------------------


## Step 5: Multi-Horizon Returns (9 features)

Compute returns over all 9 horizons: 1d, 2d, 3d, 5d, 10d, 20d, 60d, 120d, 252d

In [8]:
print("Step 5: Multi-Horizon Returns")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

for h in RETURN_HORIZONS:
    feat_name = f'Return_{h}d'
    df[feat_name] = df.groupby('Ticker')['Close'].pct_change(h).astype('float32') * 100

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Returns',
        'Description': f'{h}-day cumulative return',
        'Calculation': f'(Close_t / Close_{{t-{h}}} - 1) × 100',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Yes' if h in [1, 5, 20, 60] else 'Optional',
        'Notes': 'Short horizons (1-3d) capture reversal; long (60-252d) capture trends'
    })
    print(f"  ✓ Return_{h}d")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns now: {len(df.columns)}")
print("-"*80)

Step 5: Multi-Horizon Returns
--------------------------------------------------------------------------------
  ✓ Return_1d
  ✓ Return_2d
  ✓ Return_3d
  ✓ Return_5d
  ✓ Return_10d
  ✓ Return_20d
  ✓ Return_60d
  ✓ Return_120d
  ✓ Return_252d

✅ Added 9 features in 7.8s
   Total columns now: 19
--------------------------------------------------------------------------------


## Step 6: Multi-Horizon Volatility (6 features)

Standard deviation of daily returns over each horizon. **Skipping 1d/2d/3d** (mathematically noisy with so few data points).

In [9]:
print("Step 6: Multi-Horizon Volatility")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

for h in VOL_HORIZONS:
    feat_name = f'Volatility_{h}d'
    df[feat_name] = df.groupby('Ticker')['Return_1d'].transform(
        lambda x: x.rolling(h, min_periods=max(3, h//2)).std()
    ).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Volatility',
        'Description': f'Standard deviation of daily returns over {h} days',
        'Calculation': f'std(Return_1d) over rolling {h}-day window',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Yes' if h in [20, 60] else 'Optional',
        'Notes': 'Short horizons (5-10d) more noisy; longer horizons more stable'
    })
    print(f"  ✓ Volatility_{h}d")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 6: Multi-Horizon Volatility
--------------------------------------------------------------------------------
  ✓ Volatility_5d
  ✓ Volatility_10d
  ✓ Volatility_20d
  ✓ Volatility_60d
  ✓ Volatility_120d
  ✓ Volatility_252d

✅ Added 6 features in 3.3s
   Total columns: 25
--------------------------------------------------------------------------------


## Step 7: Risk-Adjusted Returns — Sharpe, Sortino, Calmar (16 features)

**Why these matter:** Your conviction labels (40%+, 20-40%, 10-20%) are essentially risk-adjusted return targets. These features directly proxy what you're predicting.

- **Sharpe** = annualized return / annualized vol (uses total vol)
- **Sortino** = annualized return / downside vol (penalizes only negative vol)
- **Calmar** = annualized return / max drawdown (4 horizons only — needs ≥20 days)

In [10]:
print("Step 7: Risk-Adjusted Returns (Sharpe, Sortino, Calmar)")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# --- Sharpe Ratio ---
# Annualized Sharpe = (mean daily return / daily std) × sqrt(252)
print("Computing Sharpe ratios...")
for h in VOL_HORIZONS:
    feat_name = f'Sharpe_{h}d'
    # FIX V5: add min_periods (consistent with Volatility step) so a single
    # mid-window NaN in Return_1d cannot blank the whole window.
    df[feat_name] = df.groupby('Ticker')['Return_1d'].transform(
        lambda x: (x.rolling(h, min_periods=max(3, h//2)).mean() /
                   x.rolling(h, min_periods=max(3, h//2)).std()) * np.sqrt(252)
    ).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Risk-Adjusted',
        'Description': f'Annualized Sharpe ratio over {h} days',
        'Calculation': f'(mean(Return_1d) / std(Return_1d)) × sqrt(252) over {h} days',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Yes' if h in [20, 60] else 'Optional',
        'Notes': 'Directly proxies conviction labels (risk-adjusted returns)'
    })
    print(f"  ✓ Sharpe_{h}d")

# --- Sortino Ratio ---
# Uses downside deviation (vol of negative returns only)
print("\nComputing Sortino ratios...")
for h in VOL_HORIZONS:
    feat_name = f'Sortino_{h}d'

    def compute_sortino(returns, window=h):
        # Downside: min(0, r)^2
        # FIX V5: add min_periods for consistency / gap tolerance.
        mp = max(3, window // 2)
        downside_sq = (returns.clip(upper=0)) ** 2
        downside_var = downside_sq.rolling(window, min_periods=mp).mean()
        downside_std = np.sqrt(downside_var)
        rolling_mean = returns.rolling(window, min_periods=mp).mean()
        return (rolling_mean / downside_std) * np.sqrt(252)

    df[feat_name] = df.groupby('Ticker')['Return_1d'].transform(compute_sortino).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Risk-Adjusted',
        'Description': f'Annualized Sortino ratio over {h} days',
        'Calculation': f'(mean(Return_1d) / downside_std) × sqrt(252) over {h} days',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Optional',
        'Notes': 'Like Sharpe but only penalizes downside vol'
    })
    print(f"  ✓ Sortino_{h}d")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features (Sharpe + Sortino) in {time.time()-step_start:.1f}s")
print(f"   Calmar will be added after Max Drawdown features (Step 8)")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 7: Risk-Adjusted Returns (Sharpe, Sortino, Calmar)
--------------------------------------------------------------------------------
Computing Sharpe ratios...
  ✓ Sharpe_5d
  ✓ Sharpe_10d
  ✓ Sharpe_20d
  ✓ Sharpe_60d
  ✓ Sharpe_120d
  ✓ Sharpe_252d

Computing Sortino ratios...
  ✓ Sortino_5d
  ✓ Sortino_10d
  ✓ Sortino_20d
  ✓ Sortino_60d
  ✓ Sortino_120d
  ✓ Sortino_252d

✅ Added 12 features (Sharpe + Sortino) in 12.9s
   Calmar will be added after Max Drawdown features (Step 8)
   Total columns: 37
--------------------------------------------------------------------------------


## Step 8: Drawdown Features (12 features)

**Why critical:** Your exit triggers at 1.5x ATR drawdown from peak. Past drawdown patterns predict future stop-out probability.

- **MaxDrawdown_Hd** = worst peak-to-trough loss in H days
- **DaysSincePeak_Hd** = days since rolling H-day high

In [11]:
print("Step 8: Drawdown Features")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

def max_drawdown_rolling(close, window):
    """Worst peak-to-trough decline WITHIN each rolling window (positive %).
    FIX V5: single-pass calculation. Previously used a double rolling window
    (rolling_max then rolling_min) which delayed the first valid value to day
    2*window (e.g. MaxDrawdown_252d started at day 504 ~ July 2011). Now the
    first valid value correctly appears at day `window`."""
    def calc_mdd(prices):
        # prices is a numpy array of length `window` (raw=True)
        running_max = np.maximum.accumulate(prices)
        drawdowns = (prices - running_max) / running_max * 100
        return abs(drawdowns.min())
    return close.rolling(window).apply(calc_mdd, raw=True)

def days_since_peak_rolling(close, window):
    """For each point, days since the max in last `window` days"""
    def find_days_since_max(x):
        if len(x) < 1:
            return np.nan
        return len(x) - 1 - np.argmax(x)
    return close.rolling(window).apply(find_days_since_max, raw=True)

print("Computing Max Drawdown features...")
for h in VOL_HORIZONS:
    feat_name = f'MaxDrawdown_{h}d'
    df[feat_name] = df.groupby('Ticker')['Close'].transform(
        lambda x: max_drawdown_rolling(x, h)
    ).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Drawdown',
        'Description': f'Maximum peak-to-trough drawdown in last {h} days',
        'Calculation': f'abs(min((Close - rolling_max(Close, {h})) / rolling_max(Close, {h}) × 100))',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Yes' if h == 60 else 'Optional',
        'Notes': 'Direct proxy for stop-out risk in 1.5x ATR strategy'
    })
    print(f"  ✓ MaxDrawdown_{h}d")

print("\nComputing Days Since Peak (may be slower)...")
for h in VOL_HORIZONS:
    feat_name = f'DaysSincePeak_{h}d'
    df[feat_name] = df.groupby('Ticker')['Close'].transform(
        lambda x: days_since_peak_rolling(x, h)
    ).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Drawdown',
        'Description': f'Days since the {h}-day rolling peak',
        'Calculation': f'For each row, position of max Close in last {h} days',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Optional',
        'Notes': 'Captures momentum freshness; long time since peak = stale'
    })
    print(f"  ✓ DaysSincePeak_{h}d")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")

# Now add Calmar ratio (needs MaxDrawdown)
print("\nNow adding Calmar ratios...")
step2_start = time.time()
cols_before2 = len(df.columns)

for h in [20, 60, 120, 252]:  # Need ≥20 for stable DD
    feat_name = f'Calmar_{h}d'
    # Calmar = annualized return / max drawdown
    # Annualized return = mean daily return × 252
    rolling_mean_ret = df.groupby('Ticker')['Return_1d'].transform(
        lambda x: x.rolling(h).mean() * 252
    )
    df[feat_name] = (rolling_mean_ret / df[f'MaxDrawdown_{h}d']).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Risk-Adjusted',
        'Description': f'Calmar ratio over {h} days',
        'Calculation': f'(mean(Return_1d) × 252) / MaxDrawdown_{h}d',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Optional',
        'Notes': 'Return per unit of drawdown — perfect match for ATR exit strategy'
    })
    print(f"  ✓ Calmar_{h}d")

cols_added2 = len(df.columns) - cols_before2
print(f"\n✅ Added {cols_added2} Calmar features in {time.time()-step2_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

# Memory cleanup
gc.collect()

Step 8: Drawdown Features
--------------------------------------------------------------------------------
Computing Max Drawdown features...
  ✓ MaxDrawdown_5d
  ✓ MaxDrawdown_10d
  ✓ MaxDrawdown_20d
  ✓ MaxDrawdown_60d
  ✓ MaxDrawdown_120d
  ✓ MaxDrawdown_252d

Computing Days Since Peak (may be slower)...
  ✓ DaysSincePeak_5d
  ✓ DaysSincePeak_10d
  ✓ DaysSincePeak_20d
  ✓ DaysSincePeak_60d
  ✓ DaysSincePeak_120d
  ✓ DaysSincePeak_252d

✅ Added 12 features in 176.4s
   Total columns: 49

Now adding Calmar ratios...
  ✓ Calmar_20d
  ✓ Calmar_60d
  ✓ Calmar_120d
  ✓ Calmar_252d

✅ Added 4 Calmar features in 2.6s
   Total columns: 53
--------------------------------------------------------------------------------


0

## Step 9: Up-Days Percentage (6 features)

Percentage of days with positive returns. High % = consistent winners.

In [12]:
print("Step 9: Up-Days Percentage")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# Create binary up-day indicator (will be used in rolling)
df['_up_day_temp'] = (df['Return_1d'] > 0).astype('float32')

for h in VOL_HORIZONS:
    feat_name = f'UpDays_Pct_{h}d'
    df[feat_name] = (df.groupby('Ticker')['_up_day_temp'].transform(
        lambda x: x.rolling(h).mean()
    ) * 100).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Up-Days',
        'Description': f'Percentage of up days in last {h} days',
        'Calculation': f'sum(Return_1d > 0) / {h} × 100',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Optional',
        'Notes': 'Trend persistence — high quality momentum has high up-day ratio'
    })
    print(f"  ✓ UpDays_Pct_{h}d")

# Drop temp column
df = df.drop(columns=['_up_day_temp'])

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 9: Up-Days Percentage
--------------------------------------------------------------------------------
  ✓ UpDays_Pct_5d
  ✓ UpDays_Pct_10d
  ✓ UpDays_Pct_20d
  ✓ UpDays_Pct_60d
  ✓ UpDays_Pct_120d
  ✓ UpDays_Pct_252d

✅ Added 6 features in 4.4s
   Total columns: 59
--------------------------------------------------------------------------------


## Step 10: Returns Distribution — Skewness & Kurtosis (8 features)

- **Skewness:** Negative = downside tail risk; Positive = upside potential
- **Kurtosis:** High = fat tails, extreme moves likely

Only 20d+ horizons (need ≥20 points for stable estimates).

In [13]:
print("Step 10: Returns Distribution (Skewness, Kurtosis)")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

print("Computing Skewness...")
for h in DISTRIBUTION_HORIZONS:
    feat_name = f'Skew_{h}d'
    df[feat_name] = df.groupby('Ticker')['Return_1d'].transform(
        lambda x: x.rolling(h).skew()
    ).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Distribution',
        'Description': f'Skewness of daily returns over {h} days',
        'Calculation': f'rolling skewness of Return_1d over {h} days',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'No',
        'Notes': 'Negative skew = crash risk; positive = upside potential'
    })
    print(f"  ✓ Skew_{h}d")

print("\nComputing Kurtosis...")
for h in DISTRIBUTION_HORIZONS:
    feat_name = f'Kurtosis_{h}d'
    df[feat_name] = df.groupby('Ticker')['Return_1d'].transform(
        lambda x: x.rolling(h).kurt()
    ).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Distribution',
        'Description': f'Kurtosis of daily returns over {h} days',
        'Calculation': f'rolling kurtosis of Return_1d over {h} days',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'No',
        'Notes': 'Fat tails (high kurtosis) = extreme moves likely'
    })
    print(f"  ✓ Kurtosis_{h}d")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 10: Returns Distribution (Skewness, Kurtosis)
--------------------------------------------------------------------------------
Computing Skewness...
  ✓ Skew_20d
  ✓ Skew_60d
  ✓ Skew_120d
  ✓ Skew_252d

Computing Kurtosis...
  ✓ Kurtosis_20d
  ✓ Kurtosis_60d
  ✓ Kurtosis_120d
  ✓ Kurtosis_252d

✅ Added 8 features in 4.3s
   Total columns: 67
--------------------------------------------------------------------------------


## Step 11: Multi-Horizon Volume Ratios (6 features)

`Volume_Ratio_Hd` = today's volume / average volume over last H days. Spike = institutional activity.

In [14]:
print("Step 11: Multi-Horizon Volume Ratios")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

for h in VOL_HORIZONS:
    feat_name = f'Volume_Ratio_{h}d'
    rolling_vol_mean = df.groupby('Ticker')['Volume'].transform(
        lambda x: x.rolling(h).mean()
    )
    df[feat_name] = (df['Volume'] / rolling_vol_mean).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Volume',
        'Description': f'Today\'s volume / {h}-day average volume',
        'Calculation': f'Volume_t / mean(Volume, {h})',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Yes' if h == 20 else 'Optional',
        'Notes': 'Volume spike (>2) often signals institutional activity'
    })
    print(f"  ✓ Volume_Ratio_{h}d")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 11: Multi-Horizon Volume Ratios
--------------------------------------------------------------------------------
  ✓ Volume_Ratio_5d
  ✓ Volume_Ratio_10d
  ✓ Volume_Ratio_20d
  ✓ Volume_Ratio_60d
  ✓ Volume_Ratio_120d
  ✓ Volume_Ratio_252d

✅ Added 6 features in 3.2s
   Total columns: 73
--------------------------------------------------------------------------------


## Step 12: Technical Indicators — RSI, MACD, BB, ATR, MFI, ADX (12 features)

**Note:** `ATR_Pct` is the most critical feature — directly used in your 1.5x ATR exit strategy.

In [15]:
print("Step 12: Technical Indicators")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

def calc_technical_indicators(group):
    """Calculate all technical indicators for one stock"""
    # RSI
    group['RSI_14'] = ta.momentum.RSIIndicator(
        close=group['Close'], window=RSI_PERIOD
    ).rsi()

    # MACD
    macd = ta.trend.MACD(close=group['Close'])
    group['MACD'] = macd.macd()
    group['MACD_Signal'] = macd.macd_signal()
    group['MACD_Hist'] = macd.macd_diff()

    # Bollinger Bands (only Width and Position)
    bb = ta.volatility.BollingerBands(
        close=group['Close'], window=BB_PERIOD, window_dev=BB_STD
    )
    bb_upper = bb.bollinger_hband()
    bb_lower = bb.bollinger_lband()
    bb_middle = bb.bollinger_mavg()
    group['BB_Width'] = (bb_upper - bb_lower) / bb_middle * 100
    group['BB_Position'] = (group['Close'] - bb_lower) / (bb_upper - bb_lower)

    # ATR
    atr = ta.volatility.AverageTrueRange(
        high=group['High'], low=group['Low'], close=group['Close'], window=ATR_PERIOD
    ).average_true_range()
    group['ATR_14'] = atr
    group['ATR_Pct'] = (atr / group['Close']) * 100

    # MFI
    try:
        group['MFI_14'] = ta.volume.MFIIndicator(
            high=group['High'], low=group['Low'], close=group['Close'],
            volume=group['Volume'], window=MFI_PERIOD
        ).money_flow_index()
    except:
        group['MFI_14'] = np.nan

    # ADX (trend strength)
    adx = ta.trend.ADXIndicator(
        high=group['High'], low=group['Low'], close=group['Close'], window=ADX_PERIOD
    )
    group['ADX_14'] = adx.adx()
    group['DI_Plus_14'] = adx.adx_pos()
    group['DI_Minus_14'] = adx.adx_neg()

    return group

print("Computing technical indicators (this may take a few minutes)...")
df = df.groupby('Ticker', group_keys=False).apply(calc_technical_indicators)

# Convert to float32
ti_cols = ['RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Width', 'BB_Position',
           'ATR_14', 'ATR_Pct', 'MFI_14', 'ADX_14', 'DI_Plus_14', 'DI_Minus_14']
for c in ti_cols:
    df[c] = df[c].astype('float32')

# Document features
ti_metadata = [
    ('RSI_14', 'Technical', 'Relative Strength Index (14-day)', 'ta.momentum.RSIIndicator(Close, 14)', 14, 'Yes', 'Yes', 'Overbought >70, Oversold <30'),
    ('MACD', 'Technical', 'MACD line (12-26 EMA difference)', 'EMA(Close, 12) - EMA(Close, 26)', None, 'Yes', 'Yes', 'Trend change indicator'),
    ('MACD_Signal', 'Technical', 'MACD signal line (9-day EMA of MACD)', 'EMA(MACD, 9)', None, 'Yes', 'Optional', 'Smoothed MACD signal'),
    ('MACD_Hist', 'Technical', 'MACD histogram (MACD - Signal)', 'MACD - MACD_Signal', None, 'Yes', 'Yes', 'Leading indicator — most useful of MACD trio'),
    ('BB_Width', 'Technical', 'Bollinger Band width %', '(BB_Upper - BB_Lower) / BB_Middle × 100', 20, 'Yes', 'Yes', 'Volatility squeeze/expansion'),
    ('BB_Position', 'Technical', 'Position within Bollinger Bands (0-1)', '(Close - BB_Lower) / (BB_Upper - BB_Lower)', 20, 'Yes', 'Yes', '0 = at lower band, 1 = at upper band'),
    ('ATR_14', 'Technical', 'Average True Range (14-day)', 'ta.volatility.AverageTrueRange(High, Low, Close, 14)', 14, 'Yes', 'Yes', 'Absolute volatility in price units'),
    ('ATR_Pct', 'Technical', 'ATR as % of Close', 'ATR_14 / Close × 100', 14, 'Yes', 'Yes', 'CRITICAL — directly used in 1.5x ATR exit strategy'),
    ('MFI_14', 'Technical', 'Money Flow Index (14-day)', 'Volume-weighted RSI', 14, 'Yes', 'Optional', 'Combines price and volume'),
    ('ADX_14', 'Technical', 'Average Directional Index (trend strength)', 'ta.trend.ADXIndicator(High, Low, Close, 14)', 14, 'Yes', 'Yes', '>25 = strong trend, <20 = ranging market'),
    ('DI_Plus_14', 'Technical', 'Positive directional indicator', 'Bullish pressure', 14, 'Yes', 'Optional', 'Component of ADX'),
    ('DI_Minus_14', 'Technical', 'Negative directional indicator', 'Bearish pressure', 14, 'Yes', 'Optional', 'Component of ADX'),
]

for name, cat, desc, calc, horizon, xgb, lstm, notes in ti_metadata:
    feature_metadata.append({
        'Feature_Name': name, 'Category': cat, 'Description': desc,
        'Calculation': calc, 'Horizon_Days': horizon,
        'Use_for_XGBoost': xgb, 'Use_for_LSTM': lstm, 'Notes': notes
    })

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

gc.collect()

Step 12: Technical Indicators
--------------------------------------------------------------------------------
Computing technical indicators (this may take a few minutes)...

✅ Added 12 features in 109.1s
   Total columns: 85
--------------------------------------------------------------------------------


0

## Step 13: Moving Averages & Price-MA Ratios (7 features)

SMA20/50/200 and `Price_vs_SMA` ratios (the ratios are what matters — raw MAs are scale-dependent).

In [16]:
print("Step 13: Moving Averages & Price Ratios")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# SMAs
for window in [20, 50, 200]:
    feat_name = f'SMA_{window}'
    df[feat_name] = df.groupby('Ticker')['Close'].transform(
        lambda x: x.rolling(window).mean()
    ).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Moving Average',
        'Description': f'{window}-day Simple Moving Average',
        'Calculation': f'mean(Close, {window})',
        'Horizon_Days': window,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'No',
        'Notes': 'Raw MA — for LSTM, use Price_vs_SMA ratios instead'
    })
    print(f"  ✓ SMA_{window}")

# Price vs SMA ratios (LSTM-friendly)
for window in [20, 50, 200]:
    feat_name = f'Price_vs_SMA{window}'
    df[feat_name] = ((df['Close'] - df[f'SMA_{window}']) / df[f'SMA_{window}'] * 100).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Moving Average',
        'Description': f'Position relative to {window}-day SMA (%)',
        'Calculation': f'(Close - SMA_{window}) / SMA_{window} × 100',
        'Horizon_Days': window,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Yes',
        'Notes': 'Scale-free, suitable for LSTM normalization'
    })
    print(f"  ✓ Price_vs_SMA{window}")

# MA Alignment (1 if perfect uptrend: SMA20 > SMA50 > SMA200)
df['MA_Alignment'] = (
    (df['SMA_20'] > df['SMA_50']) & (df['SMA_50'] > df['SMA_200'])
).astype('int8')
feature_metadata.append({
    'Feature_Name': 'MA_Alignment',
    'Category': 'Moving Average',
    'Description': '1 if SMA20 > SMA50 > SMA200 (perfect uptrend)',
    'Calculation': '(SMA_20 > SMA_50) AND (SMA_50 > SMA_200)',
    'Horizon_Days': 200,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Optional',
    'Notes': 'Binary signal for strong long-term uptrend'
})
print(f"  ✓ MA_Alignment")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 13: Moving Averages & Price Ratios
--------------------------------------------------------------------------------
  ✓ SMA_20
  ✓ SMA_50
  ✓ SMA_200
  ✓ Price_vs_SMA20
  ✓ Price_vs_SMA50
  ✓ Price_vs_SMA200
  ✓ MA_Alignment

✅ Added 7 features in 1.6s
   Total columns: 92
--------------------------------------------------------------------------------


## Step 14: 52-Week Features (4 features)

Psychological support/resistance levels. Stocks near 52W highs have momentum continuation tendency.

In [17]:
print("Step 14: 52-Week Features")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# 52-week high and low (rolling)
df['_52W_High'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(252).max())
df['_52W_Low'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(252).min())

# Pct from 52W high (will be ≤0)
df['Pct_From_52W_High'] = ((df['Close'] - df['_52W_High']) / df['_52W_High'] * 100).astype('float32')
feature_metadata.append({
    'Feature_Name': 'Pct_From_52W_High',
    'Category': '52-Week',
    'Description': 'Percentage distance from 52-week high (≤0)',
    'Calculation': '(Close - 52W_High) / 52W_High × 100',
    'Horizon_Days': 252,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Yes',
    'Notes': 'Psychological resistance level — stocks at 52W high often continue'
})
print("  ✓ Pct_From_52W_High")

# Pct from 52W low (will be ≥0)
df['Pct_From_52W_Low'] = ((df['Close'] - df['_52W_Low']) / df['_52W_Low'] * 100).astype('float32')
feature_metadata.append({
    'Feature_Name': 'Pct_From_52W_Low',
    'Category': '52-Week',
    'Description': 'Percentage distance from 52-week low (≥0)',
    'Calculation': '(Close - 52W_Low) / 52W_Low × 100',
    'Horizon_Days': 252,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Yes',
    'Notes': 'Distance from value zone'
})
print("  ✓ Pct_From_52W_Low")

# Position in 52W range (0-1 scale)
df['Position_in_52W_Range'] = (
    (df['Close'] - df['_52W_Low']) / (df['_52W_High'] - df['_52W_Low'])
).astype('float32')
feature_metadata.append({
    'Feature_Name': 'Position_in_52W_Range',
    'Category': '52-Week',
    'Description': 'Position in 52-week range (0-1 scale)',
    'Calculation': '(Close - 52W_Low) / (52W_High - 52W_Low)',
    'Horizon_Days': 252,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Yes',
    'Notes': '0 = at year low, 1 = at year high'
})
print("  ✓ Position_in_52W_Range")

# Days since 52W high
def days_since_max_252(close):
    def find_days_since_max(x):
        return len(x) - 1 - np.argmax(x)
    return close.rolling(252).apply(find_days_since_max, raw=True)

df['Days_Since_52W_High'] = df.groupby('Ticker')['Close'].transform(
    days_since_max_252
).astype('float32')
feature_metadata.append({
    'Feature_Name': 'Days_Since_52W_High',
    'Category': '52-Week',
    'Description': 'Days since the 52-week high',
    'Calculation': 'Position of max Close in last 252 days',
    'Horizon_Days': 252,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Optional',
    'Notes': 'Momentum freshness — recent high = strong'
})
print("  ✓ Days_Since_52W_High")

# Drop temp columns
df = df.drop(columns=['_52W_High', '_52W_Low'])

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

gc.collect()

Step 14: 52-Week Features
--------------------------------------------------------------------------------
  ✓ Pct_From_52W_High
  ✓ Pct_From_52W_Low
  ✓ Position_in_52W_Range
  ✓ Days_Since_52W_High

✅ Added 4 features in 8.9s
   Total columns: 96
--------------------------------------------------------------------------------


0

## Step 15: Higher Highs / Lower Lows Counts (12 features)

Classic technical analysis. Count days where High > previous day's High (or Low < previous Low) in a rolling window.

In [18]:
print("Step 15: Higher Highs / Lower Lows")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# Create daily indicators
df['_hh_indicator'] = df.groupby('Ticker')['High'].transform(
    lambda x: (x > x.shift(1)).astype('float32')
)
df['_ll_indicator'] = df.groupby('Ticker')['Low'].transform(
    lambda x: (x < x.shift(1)).astype('float32')
)

for h in HHLL_HORIZONS:
    # Higher highs count
    hh_name = f'Higher_Highs_Count_{h}d'
    df[hh_name] = df.groupby('Ticker')['_hh_indicator'].transform(
        lambda x: x.rolling(h).sum()
    ).astype('float32')
    feature_metadata.append({
        'Feature_Name': hh_name,
        'Category': 'HH/LL',
        'Description': f'Count of higher-high days in last {h} days',
        'Calculation': f'sum(High_t > High_{{t-1}}) over {h} days',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'No',
        'Notes': 'Strong uptrend = consistent higher highs'
    })
    print(f"  ✓ Higher_Highs_Count_{h}d")

    # Lower lows count
    ll_name = f'Lower_Lows_Count_{h}d'
    df[ll_name] = df.groupby('Ticker')['_ll_indicator'].transform(
        lambda x: x.rolling(h).sum()
    ).astype('float32')
    feature_metadata.append({
        'Feature_Name': ll_name,
        'Category': 'HH/LL',
        'Description': f'Count of lower-low days in last {h} days',
        'Calculation': f'sum(Low_t < Low_{{t-1}}) over {h} days',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'No',
        'Notes': 'Downtrend signal'
    })
    print(f"  ✓ Lower_Lows_Count_{h}d")

    # HH/LL Ratio
    hhll_name = f'HH_Ratio_{h}d'
    df[hhll_name] = (df[hh_name] / (df[hh_name] + df[ll_name])).astype('float32')
    feature_metadata.append({
        'Feature_Name': hhll_name,
        'Category': 'HH/LL',
        'Description': f'Higher highs ratio: HH / (HH + LL) over {h} days',
        'Calculation': f'Higher_Highs_Count_{h}d / (Higher_Highs_Count_{h}d + Lower_Lows_Count_{h}d)',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Optional',
        'Notes': '>0.5 = net uptrend, <0.5 = net downtrend'
    })
    print(f"  ✓ HH_Ratio_{h}d")

# Drop temp
df = df.drop(columns=['_hh_indicator', '_ll_indicator'])

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 15: Higher Highs / Lower Lows
--------------------------------------------------------------------------------
  ✓ Higher_Highs_Count_10d
  ✓ Lower_Lows_Count_10d
  ✓ HH_Ratio_10d
  ✓ Higher_Highs_Count_20d
  ✓ Lower_Lows_Count_20d
  ✓ HH_Ratio_20d
  ✓ Higher_Highs_Count_60d
  ✓ Lower_Lows_Count_60d
  ✓ HH_Ratio_60d
  ✓ Higher_Highs_Count_120d
  ✓ Lower_Lows_Count_120d
  ✓ HH_Ratio_120d

✅ Added 12 features in 6.4s
   Total columns: 108
--------------------------------------------------------------------------------


## Step 16: Volume Indicators — OBV, CMF, Value Traded (5 features)

In [19]:
print("Step 16: Volume Indicators")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

def calc_volume_indicators(group):
    # OBV (On-Balance Volume)
    group['OBV'] = ta.volume.OnBalanceVolumeIndicator(
        close=group['Close'], volume=group['Volume']
    ).on_balance_volume()

    # OBV vs its 20-day MA
    obv_ma = group['OBV'].rolling(20).mean()
    group['OBV_MA_Ratio'] = group['OBV'] / obv_ma

    # Chaikin Money Flow
    try:
        group['CMF_20'] = ta.volume.ChaikinMoneyFlowIndicator(
            high=group['High'], low=group['Low'], close=group['Close'],
            volume=group['Volume'], window=CMF_PERIOD
        ).chaikin_money_flow()
    except:
        group['CMF_20'] = np.nan

    return group

print("Computing OBV, CMF...")
df = df.groupby('Ticker', group_keys=False).apply(calc_volume_indicators)

# Value traded (Close × Volume)
df['Value_Traded'] = (df['Close'] * df['Volume']).astype('float32')

# Log of 20-day average value traded (LSTM-friendly liquidity measure)
df['Value_Traded_MA20_Log'] = df.groupby('Ticker')['Value_Traded'].transform(
    lambda x: np.log1p(x.rolling(20).mean())
).astype('float32')

# Convert to float32
for c in ['OBV', 'OBV_MA_Ratio', 'CMF_20']:
    df[c] = df[c].astype('float32')

# Document
vol_meta = [
    ('OBV', 'Volume', 'On-Balance Volume', 'Cumulative volume on up vs down days', None, 'Yes', 'No', 'Raw OBV is scale-dependent'),
    ('OBV_MA_Ratio', 'Volume', 'OBV / 20-day OBV average', 'OBV / mean(OBV, 20)', 20, 'Yes', 'Yes', 'OBV momentum — LSTM-friendly scale'),
    ('CMF_20', 'Volume', 'Chaikin Money Flow (20-day)', 'Buying vs selling pressure', 20, 'Yes', 'Optional', 'Positive = accumulation, negative = distribution'),
    ('Value_Traded', 'Volume', 'INR value traded today', 'Close × Volume', None, 'Yes', 'No', 'Raw value — scale-dependent'),
    ('Value_Traded_MA20_Log', 'Volume', 'Log of 20-day avg value traded', 'log(1 + mean(Value_Traded, 20))', 20, 'Yes', 'Yes', 'Liquidity tier — scale-normalized'),
]
for name, cat, desc, calc, horizon, xgb, lstm, notes in vol_meta:
    feature_metadata.append({
        'Feature_Name': name, 'Category': cat, 'Description': desc,
        'Calculation': calc, 'Horizon_Days': horizon,
        'Use_for_XGBoost': xgb, 'Use_for_LSTM': lstm, 'Notes': notes
    })
    print(f"  ✓ {name}")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

gc.collect()

Step 16: Volume Indicators
--------------------------------------------------------------------------------
Computing OBV, CMF...
  ✓ OBV
  ✓ OBV_MA_Ratio
  ✓ CMF_20
  ✓ Value_Traded
  ✓ Value_Traded_MA20_Log

✅ Added 4 features in 6.8s
   Total columns: 112
--------------------------------------------------------------------------------


0

## Step 17: Daily Price Action Features (4 features)

In [20]:
print("Step 17: Daily Price Action")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# High-Low spread as % of close
df['HL_Spread_Pct'] = ((df['High'] - df['Low']) / df['Close'] * 100).astype('float32')
feature_metadata.append({
    'Feature_Name': 'HL_Spread_Pct',
    'Category': 'Price Action',
    'Description': 'Daily High-Low spread as % of Close',
    'Calculation': '(High - Low) / Close × 100',
    'Horizon_Days': 1,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Yes',
    'Notes': 'Intraday volatility measure'
})
print("  ✓ HL_Spread_Pct")

# Close position in daily range
df['Close_in_Daily_Range'] = (
    (df['Close'] - df['Low']) / (df['High'] - df['Low'])
).astype('float32')
feature_metadata.append({
    'Feature_Name': 'Close_in_Daily_Range',
    'Category': 'Price Action',
    'Description': 'Where Close is in daily range (0-1)',
    'Calculation': '(Close - Low) / (High - Low)',
    'Horizon_Days': 1,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Yes',
    'Notes': '0 = closed at low (weak), 1 = closed at high (strong)'
})
print("  ✓ Close_in_Daily_Range")

# Overnight gap
prev_close = df.groupby('Ticker')['Close'].shift(1)
df['Overnight_Gap'] = ((df['Open'] - prev_close) / prev_close * 100).astype('float32')
feature_metadata.append({
    'Feature_Name': 'Overnight_Gap',
    'Category': 'Price Action',
    'Description': 'Overnight gap % (Open vs previous Close)',
    'Calculation': '(Open - PrevClose) / PrevClose × 100',
    'Horizon_Days': 1,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Optional',
    'Notes': 'News-driven price moves before market open'
})
print("  ✓ Overnight_Gap")

# Intraday move (Open to Close)
df['Intraday_Move'] = ((df['Close'] - df['Open']) / df['Open'] * 100).astype('float32')
feature_metadata.append({
    'Feature_Name': 'Intraday_Move',
    'Category': 'Price Action',
    'Description': 'Intraday move % (Open to Close)',
    'Calculation': '(Close - Open) / Open × 100',
    'Horizon_Days': 1,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Optional',
    'Notes': 'Trading session direction (separates overnight from intraday moves)'
})
print("  ✓ Intraday_Move")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 17: Daily Price Action
--------------------------------------------------------------------------------
  ✓ HL_Spread_Pct
  ✓ Close_in_Daily_Range
  ✓ Overnight_Gap
  ✓ Intraday_Move

✅ Added 4 features in 0.4s
   Total columns: 116
--------------------------------------------------------------------------------


## Step 18: Excess Returns vs Nifty 500 (9 features)

Pure alpha — how much stock outperforms the broad market.

In [21]:
print("Step 18: Excess Returns vs Nifty 500")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# Validate Nifty 500 column exists
if 'NIFTY_500' not in df_indices.columns:
    print("❌ NIFTY_500 column not in indices file!")
    print(f"   Available: {df_indices.columns.tolist()}")
    raise ValueError("NIFTY_500 missing")

# FIX V5: use the gap-free multi-horizon market returns from Step 4b
# (df_market_returns) instead of a direct how='left' merge that could leave
# NaN holes on dates missing from the index.
n500_cols = [f'Market_Return_{h}d' for h in RETURN_HORIZONS]
df = df.merge(df_market_returns[['Date'] + n500_cols], on='Date', how='left')

# Compute excess returns
for h in RETURN_HORIZONS:
    feat_name = f'Excess_Ret_N500_{h}d'
    df[feat_name] = (df[f'Return_{h}d'] - df[f'Market_Return_{h}d']).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Index-Relative',
        'Description': f'{h}-day excess return vs Nifty 500',
        'Calculation': f'Return_{h}d - Nifty500_Return_{h}d (gap-free market series)',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Yes' if h in [5, 20] else 'Optional',
        'Notes': 'Pure alpha — outperformance vs broad market'
    })
    print(f"  ✓ Excess_Ret_N500_{h}d")

# Drop the multi-horizon market returns (only needed for computation)
df = df.drop(columns=n500_cols)

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 18: Excess Returns vs Nifty 500
--------------------------------------------------------------------------------
  ✓ Excess_Ret_N500_1d
  ✓ Excess_Ret_N500_2d
  ✓ Excess_Ret_N500_3d
  ✓ Excess_Ret_N500_5d
  ✓ Excess_Ret_N500_10d
  ✓ Excess_Ret_N500_20d
  ✓ Excess_Ret_N500_60d
  ✓ Excess_Ret_N500_120d
  ✓ Excess_Ret_N500_252d

✅ Added 9 features in 4.3s
   Total columns: 125
--------------------------------------------------------------------------------


## Step 19: Excess Returns vs Sector (9 features)

Within-sector alpha. Picks best stocks in good sectors.

In [22]:
# Merge Sector from stock_index_mapping (needed before Step 19)
df_sector = df_mapping[['Symbol', 'Sector']].copy()
df_sector.columns = ['Ticker', 'Sector']
df = df.merge(df_sector, on='Ticker', how='left')
df['Sector'] = df['Sector'].fillna('Unknown')
print(f"✅ Sector merged. Unique sectors: {df['Sector'].nunique()}")
print(df['Sector'].value_counts())

✅ Sector merged. Unique sectors: 12
Sector
Industrials               494939
Basic Materials           414295
Consumer Cyclical         324677
Financial Services        299605
Healthcare                205751
Consumer Defensive        198181
Technology                153085
Communication Services     84117
Utilities                  80503
Energy                     76915
Real Estate                71794
Unknown                    13798
Name: count, dtype: int64


In [23]:
print("Step 19: Excess Returns vs Sector")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# Validate Sector column
if 'Sector' not in df.columns:
    print("❌ Sector column missing!")
    raise ValueError("Sector column required")

# Fill missing sectors
df['Sector'] = df['Sector'].fillna('Unknown')
print(f"✅ Sector unique values: {df['Sector'].nunique()}")

# Compute sector average returns at all horizons (per Date)
print("\nComputing sector average returns...")
for h in RETURN_HORIZONS:
    sector_return = df.groupby(['Date', 'Sector'])[f'Return_{h}d'].transform('mean')

    feat_name = f'Excess_Ret_Sector_{h}d'
    df[feat_name] = (df[f'Return_{h}d'] - sector_return).astype('float32')

    feature_metadata.append({
        'Feature_Name': feat_name,
        'Category': 'Sector-Relative',
        'Description': f'{h}-day excess return vs sector average',
        'Calculation': f'Return_{h}d - mean(Return_{h}d) over same Sector and Date',
        'Horizon_Days': h,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Yes' if h in [5, 20] else 'Optional',
        'Notes': 'Within-sector alpha — pick best stocks in good sectors'
    })
    print(f"  ✓ Excess_Ret_Sector_{h}d")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

gc.collect()

Step 19: Excess Returns vs Sector
--------------------------------------------------------------------------------
✅ Sector unique values: 12

Computing sector average returns...
  ✓ Excess_Ret_Sector_1d
  ✓ Excess_Ret_Sector_2d
  ✓ Excess_Ret_Sector_3d
  ✓ Excess_Ret_Sector_5d
  ✓ Excess_Ret_Sector_10d
  ✓ Excess_Ret_Sector_20d
  ✓ Excess_Ret_Sector_60d
  ✓ Excess_Ret_Sector_120d
  ✓ Excess_Ret_Sector_252d

✅ Added 9 features in 3.6s
   Total columns: 135
--------------------------------------------------------------------------------


0

## Step 20: Beta, R², Idiosyncratic Vol — 60d & 252d (6 features)

- **Beta:** Stock's sensitivity to market
- **R²:** Fraction of variance explained by market
- **Idiosyncratic Vol:** Stock-specific risk (vol of residuals)

**Uses vectorized formulation** (correlation-based) for speed — equivalent to rolling regression.

In [24]:
print("Step 20: Beta, R², Idiosyncratic Volatility")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# FIX V5: use the gap-free Market_Return_1d from Step 4b instead of a fresh
# how='left' merge. A single NaN market day previously cascaded into a full
# window of NaN Beta for ALL stocks (the 2014 block).
df = df.merge(df_market_returns[['Date', 'Market_Return_1d']], on='Date', how='left')
df['Market_Return_1d'] = df['Market_Return_1d'].astype('float32')

print("Computing rolling Beta, R², and Idiosyncratic Vol...")

for h in BETA_HORIZONS:
    print(f"\n  Computing for {h}-day window (this is slow)...")

    def compute_beta_r2_idio(group, window=h):
        """Compute rolling beta, R², and idiosyncratic vol for one stock"""
        stock_ret = group['Return_1d']
        mkt_ret = group['Market_Return_1d']

        # Rolling cov(stock, market) and var(market)
        # FIX V5: min_periods tolerates occasional gaps so one NaN day
        # cannot blank an entire rolling window.
        mp = int(window * 0.8)
        rolling_cov = stock_ret.rolling(window, min_periods=mp).cov(mkt_ret)
        rolling_var_mkt = mkt_ret.rolling(window, min_periods=mp).var()

        # Beta = Cov / Var_market
        beta = rolling_cov / rolling_var_mkt

        # R² = correlation²
        correlation = stock_ret.rolling(window, min_periods=mp).corr(mkt_ret)
        r_squared = correlation ** 2

        # Idiosyncratic vol = total_vol × sqrt(1 - R²)
        total_vol = stock_ret.rolling(window, min_periods=mp).std()
        idio_vol = total_vol * np.sqrt(1 - r_squared)

        return pd.DataFrame({
            f'Beta_{h}d': beta.values,
            f'R_Squared_{h}d': r_squared.values,
            f'Idio_Vol_{h}d': idio_vol.values
        }, index=group.index)

    # Apply per ticker
    beta_results = df.groupby('Ticker', group_keys=False).apply(compute_beta_r2_idio)

    # Merge results back
    df[f'Beta_{h}d'] = beta_results[f'Beta_{h}d'].astype('float32')
    df[f'R_Squared_{h}d'] = beta_results[f'R_Squared_{h}d'].astype('float32')
    df[f'Idio_Vol_{h}d'] = beta_results[f'Idio_Vol_{h}d'].astype('float32')

    # Document
    for name_suffix, cat_suffix, desc in [
        ('Beta', 'Beta', f'Stock beta vs Nifty 500 over {h} days'),
        ('R_Squared', 'Beta', f'R-squared vs Nifty 500 over {h} days'),
        ('Idio_Vol', 'Beta', f'Idiosyncratic volatility over {h} days')
    ]:
        feature_metadata.append({
            'Feature_Name': f'{name_suffix}_{h}d',
            'Category': 'Beta/Risk',
            'Description': desc,
            'Calculation': {
                'Beta': f'Cov(Stock_Ret, Mkt_Ret) / Var(Mkt_Ret) over {h} days',
                'R_Squared': f'Corr(Stock_Ret, Mkt_Ret)² over {h} days',
                'Idio_Vol': f'Vol(Stock_Ret) × sqrt(1 - R²) over {h} days'
            }[name_suffix],
            'Horizon_Days': h,
            'Use_for_XGBoost': 'Yes',
            'Use_for_LSTM': 'Optional',
            'Notes': {
                'Beta': 'Market sensitivity — high beta = aggressive',
                'R_Squared': 'Fraction of variance explained by market',
                'Idio_Vol': 'Stock-specific risk (after removing market beta)'
            }[name_suffix]
        })
    print(f"  ✓ Beta_{h}d, R_Squared_{h}d, Idio_Vol_{h}d")

# Keep Market_Return_1d for now — we'll drop later if not needed

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

gc.collect()

Step 20: Beta, R², Idiosyncratic Volatility
--------------------------------------------------------------------------------
Computing rolling Beta, R², and Idiosyncratic Vol...

  Computing for 60-day window (this is slow)...
  ✓ Beta_60d, R_Squared_60d, Idio_Vol_60d

  Computing for 252-day window (this is slow)...
  ✓ Beta_252d, R_Squared_252d, Idio_Vol_252d

✅ Added 7 features in 11.3s
   Total columns: 142
--------------------------------------------------------------------------------


0

## Step 21: Cross-Sectional Ranks (11 features)

**XGBoost ONLY.** For each date, rank stocks across the full universe. Gives the model relative positioning.

**Anti-leakage:** Rank is computed per Date using only features that themselves use only past data.

In [25]:
print("Step 21: Cross-Sectional Ranks (XGBoost only)")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# Features to rank (cross-sectionally)
rank_features = {
    'Return_5d': 'Return rank (5d)',
    'Return_20d': 'Return rank (20d)',
    'Return_60d': 'Return rank (60d)',
    'Return_120d': 'Return rank (120d)',
    'Return_252d': 'Return rank (252d)',
    'Volatility_20d': 'Volatility rank',
    'ATR_Pct': 'ATR % rank',
    'RSI_14': 'RSI rank',
    'Volume_Ratio_20d': 'Volume surge rank',
    'Sharpe_20d': 'Sharpe rank',
    'Excess_Ret_N500_20d': 'Alpha rank vs Nifty 500'
}

for feat, desc in rank_features.items():
    if feat not in df.columns:
        print(f"  ⚠️  Skipping {feat} (not in dataframe)")
        continue

    rank_name = f'{feat}_Rank'
    # Percentile rank within each Date (0-1 scale)
    df[rank_name] = df.groupby('Date')[feat].rank(pct=True).astype('float32')

    feature_metadata.append({
        'Feature_Name': rank_name,
        'Category': 'Cross-Sectional',
        'Description': f'Percentile rank of {feat} across all stocks (per Date)',
        'Calculation': f'percentile rank of {feat} within each Date',
        'Horizon_Days': None,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'No',
        'Notes': 'XGBoost-only — LSTM processes per-stock sequences'
    })
    print(f"  ✓ {rank_name}")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 21: Cross-Sectional Ranks (XGBoost only)
--------------------------------------------------------------------------------
  ✓ Return_5d_Rank
  ✓ Return_20d_Rank
  ✓ Return_60d_Rank
  ✓ Return_120d_Rank
  ✓ Return_252d_Rank
  ✓ Volatility_20d_Rank
  ✓ ATR_Pct_Rank
  ✓ RSI_14_Rank
  ✓ Volume_Ratio_20d_Rank
  ✓ Sharpe_20d_Rank
  ✓ Excess_Ret_N500_20d_Rank

✅ Added 11 features in 14.3s
   Total columns: 153
--------------------------------------------------------------------------------


## Step 22: Market Regime Features (5 features)

Contextual features describing the overall market state.

In [26]:
print("Step 22: Market Regime Features")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# FIX V5: build market regime from the gap-free aligned index series.
# Reconstruct a gap-free NIFTY_500 price on the stock-date grid so the SMA200
# and rolling vols have no holes.
_all_dates = pd.DataFrame({'Date': sorted(df['Date'].unique())})
df_market = df_indices[['Date', 'NIFTY_500']].copy().sort_values('Date')
df_market = _all_dates.merge(df_market, on='Date', how='left')
df_market['NIFTY_500'] = df_market['NIFTY_500'].ffill().bfill()
df_market['_Nifty500_Ret_1d'] = df_market['NIFTY_500'].pct_change()
df_market['_Nifty500_SMA200'] = df_market['NIFTY_500'].rolling(200, min_periods=100).mean()
df_market['Market_Trend'] = (df_market['NIFTY_500'] > df_market['_Nifty500_SMA200']).astype('int8')
df_market['Market_Return_20d'] = (df_market['NIFTY_500'].pct_change(20) * 100).astype('float32')
df_market['Market_Volatility_60d'] = (df_market['_Nifty500_Ret_1d'].rolling(60, min_periods=30).std() * 100).astype('float32')
df_market['_Vol_20d'] = df_market['_Nifty500_Ret_1d'].rolling(20, min_periods=10).std()
df_market['_Vol_60d'] = df_market['_Nifty500_Ret_1d'].rolling(60, min_periods=30).std()
df_market['Market_Vol_Regime'] = (df_market['_Vol_20d'] > df_market['_Vol_60d']).astype('int8')

market_cols = ['Date', 'Market_Trend', 'Market_Vol_Regime', 'Market_Return_20d', 'Market_Volatility_60d']
df = df.merge(df_market[market_cols], on='Date', how='left')

for feat, desc, calc, notes in [
    ('Market_Trend', 'Bull/bear regime (1=above SMA200)', 'Nifty500 > rolling_mean(Nifty500, 200)', 'Bull markets are more conducive to long strategies'),
    ('Market_Vol_Regime', 'Vol regime (1=expanding)', 'Nifty500_Vol_20d > Nifty500_Vol_60d', 'Risk-on (1) vs risk-off (0)'),
    ('Market_Return_20d', 'Nifty 500 20-day return %', 'pct_change(Nifty500, 20) × 100', 'Recent market direction'),
    ('Market_Volatility_60d', 'Nifty 500 60-day volatility', 'std(daily returns) over 60d', 'Market stress indicator'),
]:
    feature_metadata.append({
        'Feature_Name': feat, 'Category': 'Market Regime', 'Description': desc,
        'Calculation': calc, 'Horizon_Days': None,
        'Use_for_XGBoost': 'Yes', 'Use_for_LSTM': 'Yes', 'Notes': notes
    })
    print(f"  ✓ {feat}")

# Breadth: % of stocks above SMA50 (per Date)
df['_above_sma50'] = (df['Close'] > df['SMA_50']).astype('float32')
breadth = df.groupby('Date')['_above_sma50'].mean().rename('Breadth_Pct_Above_SMA50') * 100
df = df.merge(breadth.astype('float32'), on='Date', how='left')
df = df.drop(columns=['_above_sma50'])

feature_metadata.append({
    'Feature_Name': 'Breadth_Pct_Above_SMA50',
    'Category': 'Market Regime',
    'Description': '% of stocks in universe trading above their 50-day SMA',
    'Calculation': 'mean(Close > SMA_50) over all stocks per Date × 100',
    'Horizon_Days': 50,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Yes',
    'Notes': 'Market breadth — high % = healthy market'
})
print("  ✓ Breadth_Pct_Above_SMA50")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

gc.collect()

Step 22: Market Regime Features
--------------------------------------------------------------------------------
  ✓ Market_Trend
  ✓ Market_Vol_Regime
  ✓ Market_Return_20d
  ✓ Market_Volatility_60d
  ✓ Breadth_Pct_Above_SMA50

✅ Added 5 features in 9.5s
   Total columns: 158
--------------------------------------------------------------------------------


0

## Step 23: Market Cap Features (4 features)

From `stock_index_mapping.csv`. **Note:** Market cap is static (one value per stock as of mapping date).

In [27]:
print("Step 23: Market Cap Features")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# Prepare mapping dataframe — handle ticker format mismatch
# data_clean has 'Ticker' (e.g., 'BCG'), mapping has 'Symbol' (e.g., 'BCG')
df_map_subset = df_mapping[['Symbol', 'Market_Cap_INR_Cr', 'Rank', 'Index_Classification']].copy()
df_map_subset.columns = ['Ticker', 'Market_Cap_INR_Cr', 'Market_Cap_Rank', 'Index_Classification']

# Merge
df = df.merge(df_map_subset, on='Ticker', how='left')

# Convert types
df['Market_Cap_INR_Cr'] = df['Market_Cap_INR_Cr'].astype('float32')
df['Market_Cap_Rank'] = df['Market_Cap_Rank'].astype('float32')

# Log market cap (LSTM-friendly)
df['Market_Cap_Log'] = np.log1p(df['Market_Cap_INR_Cr']).astype('float32')

# Check mapping coverage
missing_mcap = df['Market_Cap_INR_Cr'].isna().sum()
total_rows = len(df)
print(f"  Mapping coverage: {(1 - missing_mcap/total_rows)*100:.1f}% ({total_rows - missing_mcap:,}/{total_rows:,})")

for feat, desc, calc, xgb, lstm, notes in [
    ('Market_Cap_INR_Cr', 'Market capitalization in INR Crores', 'From stock_index_mapping.csv', 'Yes', 'No', 'Raw market cap'),
    ('Market_Cap_Rank', 'Rank by market cap (1=largest)', 'From stock_index_mapping.csv', 'Yes', 'No', 'Static rank as of mapping date'),
    ('Market_Cap_Log', 'Log-transformed market cap', 'log(1 + Market_Cap_INR_Cr)', 'Yes', 'Yes', 'Better-distributed; LSTM-friendly'),
    ('Index_Classification', 'Nifty 100 / Midcap 150 / Smallcap 250', 'From stock_index_mapping.csv', 'Yes', 'encoded', 'Categorical — needs encoding'),
]:
    feature_metadata.append({
        'Feature_Name': feat, 'Category': 'Market Cap', 'Description': desc,
        'Calculation': calc, 'Horizon_Days': None,
        'Use_for_XGBoost': xgb, 'Use_for_LSTM': lstm, 'Notes': notes
    })
    print(f"  ✓ {feat}")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 23: Market Cap Features
--------------------------------------------------------------------------------
  Mapping coverage: 99.4% (2,403,862/2,417,660)
  ✓ Market_Cap_INR_Cr
  ✓ Market_Cap_Rank
  ✓ Market_Cap_Log
  ✓ Index_Classification

✅ Added 4 features in 3.3s
   Total columns: 162
--------------------------------------------------------------------------------


## Step 24: Sector Encoding (1 feature)

Keep raw `Sector` and `Industry` strings; add label-encoded version for XGBoost convenience.

In [28]:
print("Step 24: Sector Encoding")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

# Label encode Sector
le_sector = LabelEncoder()
df['Sector_Encoded'] = le_sector.fit_transform(df['Sector'].astype(str)).astype('int8')
print(f"  ✓ Sector_Encoded ({df['Sector'].nunique()} unique sectors)")

# Label encode Index_Classification if present
if 'Index_Classification' in df.columns:
    le_idx = LabelEncoder()
    df['Index_Classification_Encoded'] = le_idx.fit_transform(
        df['Index_Classification'].astype(str)
    ).astype('int8')
    print(f"  ✓ Index_Classification_Encoded ({df['Index_Classification'].nunique()} unique values)")

    feature_metadata.append({
        'Feature_Name': 'Index_Classification_Encoded',
        'Category': 'Sector',
        'Description': 'Label-encoded Index Classification',
        'Calculation': 'LabelEncoder().fit_transform(Index_Classification)',
        'Horizon_Days': None,
        'Use_for_XGBoost': 'Yes',
        'Use_for_LSTM': 'Yes',
        'Notes': 'For LSTM, prefer embedding layer over label encoding'
    })

feature_metadata.append({
    'Feature_Name': 'Sector_Encoded',
    'Category': 'Sector',
    'Description': 'Label-encoded Sector',
    'Calculation': 'LabelEncoder().fit_transform(Sector)',
    'Horizon_Days': None,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'Yes',
    'Notes': 'For LSTM, prefer embedding layer over label encoding'
})

feature_metadata.append({
    'Feature_Name': 'Sector',
    'Category': 'Sector',
    'Description': 'Raw sector string from Yahoo Finance',
    'Calculation': 'From data_clean.csv',
    'Horizon_Days': None,
    'Use_for_XGBoost': 'Yes',
    'Use_for_LSTM': 'No',
    'Notes': 'Use Sector_Encoded for modeling'
})

if 'Industry' in df.columns:
    feature_metadata.append({
        'Feature_Name': 'Industry',
        'Category': 'Sector',
        'Description': 'Raw industry string from Yahoo Finance',
        'Calculation': 'From data_clean.csv',
        'Horizon_Days': None,
        'Use_for_XGBoost': 'Optional',
        'Use_for_LSTM': 'No',
        'Notes': 'More granular than Sector'
    })

feature_metadata.append({
    'Feature_Name': 'Index_Classification',
    'Category': 'Market Cap',
    'Description': 'Raw Index Classification string',
    'Calculation': 'From stock_index_mapping.csv',
    'Horizon_Days': None,
    'Use_for_XGBoost': 'Optional',
    'Use_for_LSTM': 'No',
    'Notes': 'Use Index_Classification_Encoded for modeling'
})

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 24: Sector Encoding
--------------------------------------------------------------------------------
  ✓ Sector_Encoded (12 unique sectors)
  ✓ Index_Classification_Encoded (4 unique values)

✅ Added 2 features in 1.2s
   Total columns: 164
--------------------------------------------------------------------------------


## Step 25: Date Features (11 features)

Raw date components (for XGBoost) + cyclical encoding (for LSTM continuity).

In [29]:
print("Step 25: Date Features")
print("-"*80)
step_start = time.time()
cols_before = len(df.columns)

df['Year'] = df['Date'].dt.year.astype('int16')
df['Month'] = df['Date'].dt.month.astype('int8')
df['Quarter'] = df['Date'].dt.quarter.astype('int8')
df['DayOfWeek'] = df['Date'].dt.dayofweek.astype('int8')

# Cyclical encoding (for LSTM)
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12).astype('float32')
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12).astype('float32')
df['DayOfWeek_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 5).astype('float32')
df['DayOfWeek_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 5).astype('float32')

# Binary calendar flags
df['Is_Month_End'] = df['Date'].dt.is_month_end.astype('int8')
df['Is_Quarter_End'] = df['Date'].dt.is_quarter_end.astype('int8')
# FY end for India: March 31 region (last 7 days of March)
df['Is_FY_End'] = ((df['Month'] == 3) & (df['Date'].dt.day >= 25)).astype('int8')

for feat, desc, xgb, lstm, notes in [
    ('Year', 'Year of date', 'Yes', 'Optional', 'May capture year-specific effects'),
    ('Month', 'Month (1-12)', 'Yes', 'No (use cyclical)', 'Seasonality — use cyclical for LSTM'),
    ('Quarter', 'Quarter (1-4)', 'Yes', 'Optional', 'Quarterly patterns'),
    ('DayOfWeek', 'Day of week (0-4)', 'Yes', 'No (use cyclical)', 'Weekly patterns'),
    ('Month_sin', 'Cyclical month (sin)', 'Optional', 'Yes', 'For LSTM — Dec→Jan continuity'),
    ('Month_cos', 'Cyclical month (cos)', 'Optional', 'Yes', 'For LSTM — Dec→Jan continuity'),
    ('DayOfWeek_sin', 'Cyclical day of week (sin)', 'Optional', 'Yes', 'For LSTM'),
    ('DayOfWeek_cos', 'Cyclical day of week (cos)', 'Optional', 'Yes', 'For LSTM'),
    ('Is_Month_End', '1 if last trading day of month', 'Yes', 'Yes', 'Window-dressing effects'),
    ('Is_Quarter_End', '1 if last trading day of quarter', 'Yes', 'Yes', 'Quarterly rebalancing'),
    ('Is_FY_End', '1 if late March (Indian FY end)', 'Yes', 'Yes', 'Indian fiscal year-end'),
]:
    feature_metadata.append({
        'Feature_Name': feat, 'Category': 'Date', 'Description': desc,
        'Calculation': 'From Date column', 'Horizon_Days': None,
        'Use_for_XGBoost': xgb, 'Use_for_LSTM': lstm, 'Notes': notes
    })
    print(f"  ✓ {feat}")

cols_added = len(df.columns) - cols_before
print(f"\n✅ Added {cols_added} features in {time.time()-step_start:.1f}s")
print(f"   Total columns: {len(df.columns)}")
print("-"*80)

Step 25: Date Features
--------------------------------------------------------------------------------
  ✓ Year
  ✓ Month
  ✓ Quarter
  ✓ DayOfWeek
  ✓ Month_sin
  ✓ Month_cos
  ✓ DayOfWeek_sin
  ✓ DayOfWeek_cos
  ✓ Is_Month_End
  ✓ Is_Quarter_End
  ✓ Is_FY_End

✅ Added 11 features in 0.8s
   Total columns: 175
--------------------------------------------------------------------------------


## Step 26: Document Base Features

Add base OHLCV columns to feature dictionary for completeness.

In [30]:
print("Documenting base features...")
print("-"*80)

base_metadata = [
    ('Date', 'Base', 'Trading date', 'From data_clean', None, 'No (identifier)', 'No (identifier)', 'Sort key'),
    ('Ticker', 'Base', 'Stock ticker (NSE symbol)', 'From data_clean', None, 'No (identifier)', 'No (identifier)', 'Sort key'),
    ('Open', 'Base', 'Opening price', 'From data_clean', None, 'Optional', 'No', 'Used in derived features'),
    ('High', 'Base', 'Daily high', 'From data_clean', None, 'Optional', 'No', 'Used in derived features'),
    ('Low', 'Base', 'Daily low', 'From data_clean', None, 'Optional', 'No', 'Used in derived features'),
    ('Close', 'Base', 'Closing price', 'From data_clean', None, 'Yes', 'Yes (normalized)', 'Primary price for returns'),
    ('Volume', 'Base', 'Shares traded', 'From data_clean', None, 'Yes', 'Yes (normalized)', 'Liquidity & conviction'),
    ('Market_Return_1d', 'Market Regime', 'Nifty 500 daily return %', 'pct_change(NIFTY_500) × 100', 1, 'Yes', 'Yes', 'Used in beta calculation'),
]

for name, cat, desc, calc, horizon, xgb, lstm, notes in base_metadata:
    feature_metadata.append({
        'Feature_Name': name, 'Category': cat, 'Description': desc,
        'Calculation': calc, 'Horizon_Days': horizon,
        'Use_for_XGBoost': xgb, 'Use_for_LSTM': lstm, 'Notes': notes
    })

print(f"✅ Documented {len(base_metadata)} base features")
print("-"*80)

Documenting base features...
--------------------------------------------------------------------------------
✅ Documented 8 base features
--------------------------------------------------------------------------------


## Step 27: Final Data Quality Check

In [31]:
print("Step 27: Final Data Quality Check")
print("="*80)

# Total rows and columns
print(f"📊 Final dataset:")
print(f"   Rows: {len(df):,}")
print(f"   Columns: {len(df.columns)}")
print(f"   Unique stocks: {df['Ticker'].nunique()}")
print(f"   Date range: {df['Date'].min()} to {df['Date'].max()}")

# Memory usage
memory_mb = df.memory_usage(deep=True).sum() / (1024**2)
print(f"\n💾 Memory: {memory_mb:.1f} MB ({memory_mb/1024:.2f} GB)")

# Missing value summary (top 20 features by NaN %)
print(f"\n📊 Top 20 features by missing % (expected for long-horizon features):")
nan_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
print(nan_pct.head(20))

print("\n📊 NaN distribution:")
print(f"   Features with 0% NaN: {(nan_pct == 0).sum()}")
print(f"   Features with <5% NaN: {(nan_pct < 5).sum()}")
print(f"   Features with 5-20% NaN: {((nan_pct >= 5) & (nan_pct < 20)).sum()}")
print(f"   Features with >20% NaN: {(nan_pct >= 20).sum()} (mostly long-horizon)")

print("="*80)

Step 27: Final Data Quality Check
📊 Final dataset:
   Rows: 2,417,660
   Columns: 175
   Unique stocks: 570
   Date range: 2007-01-02 00:00:00 to 2026-06-12 00:00:00

💾 Memory: 1955.1 MB (1.91 GB)

📊 Top 20 features by missing % (expected for long-horizon features):
Calmar_252d               5.982065
Volume_Ratio_252d         5.973296
Position_in_52W_Range     5.958572
Kurtosis_252d             5.941282
Return_252d               5.941282
Return_252d_Rank          5.941282
Skew_252d                 5.941282
Excess_Ret_N500_252d      5.941282
Excess_Ret_Sector_252d    5.941282
MaxDrawdown_252d          5.917706
UpDays_Pct_252d           5.917706
DaysSincePeak_252d        5.917706
Pct_From_52W_High         5.917706
Pct_From_52W_Low          5.917706
Days_Since_52W_High       5.917706
Idio_Vol_252d             4.779663
R_Squared_252d            4.779663
Beta_252d                 4.738880
SMA_200                   4.691727
Price_vs_SMA200           4.691727
dtype: float64

📊 NaN distributio

## Step 28: Save `data_features.csv`

In [32]:
print("Step 28: Saving data_features.parquet")
print("-"*80)
step_start = time.time()

# Sort by Ticker, Date for clean output
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(f"Writing {len(df):,} rows × {len(df.columns)} columns...")
df.to_parquet(OUTPUT_FILE_FEATURES, index=False)

import os
file_size_mb = os.path.getsize(OUTPUT_FILE_FEATURES) / (1024**2)
print(f"\n✅ Saved {OUTPUT_FILE_FEATURES}")
print(f"   File size: {file_size_mb:.1f} MB ({file_size_mb/1024:.2f} GB)")
print(f"   Write time: {time.time()-step_start:.1f}s")
print("-"*80)

Step 28: Saving data_features.parquet
--------------------------------------------------------------------------------
Writing 2,417,660 rows × 175 columns...

✅ Saved data_features.parquet
   File size: 1303.9 MB (1.27 GB)
   Write time: 34.1s
--------------------------------------------------------------------------------


## Step 29: Generate Feature Dictionary CSV

**This is your reference for downstream modeling code.** Use it to select features for XGBoost vs LSTM.

In [33]:
print("Step 29: Generating Feature Dictionary")
print("-"*80)

df_dict = pd.DataFrame(feature_metadata)

# Reorder columns
dict_cols = ['Feature_Name', 'Category', 'Description', 'Calculation',
             'Horizon_Days', 'Use_for_XGBoost', 'Use_for_LSTM', 'Notes']
df_dict = df_dict[dict_cols]

# Sort by category, then feature name
df_dict = df_dict.sort_values(['Category', 'Feature_Name']).reset_index(drop=True)

# Check: are all df columns documented?
documented = set(df_dict['Feature_Name'].tolist())
df_cols = set(df.columns.tolist())
missing_from_dict = df_cols - documented
extra_in_dict = documented - df_cols

if missing_from_dict:
    print(f"⚠️  Features in DF but not in dictionary: {missing_from_dict}")
    # Add minimal entries for any missing
    for missing_feat in missing_from_dict:
        df_dict = pd.concat([df_dict, pd.DataFrame([{
            'Feature_Name': missing_feat,
            'Category': 'Unknown',
            'Description': 'Undocumented feature',
            'Calculation': 'N/A',
            'Horizon_Days': None,
            'Use_for_XGBoost': 'Unknown',
            'Use_for_LSTM': 'Unknown',
            'Notes': 'Please document this feature'
        }])], ignore_index=True)
    df_dict = df_dict.sort_values(['Category', 'Feature_Name']).reset_index(drop=True)

if extra_in_dict:
    print(f"⚠️  Features in dictionary but not in DF: {extra_in_dict}")

# Save
df_dict.to_csv(OUTPUT_FILE_DICT, index=False)

print(f"\n✅ Saved {OUTPUT_FILE_DICT}")
print(f"   Total features documented: {len(df_dict)}")
print(f"\n📊 By Category:")
print(df_dict['Category'].value_counts())
print(f"\n📊 LSTM/XGB Usage:")
print(f"   XGBoost Yes:    {(df_dict['Use_for_XGBoost'] == 'Yes').sum()}")
print(f"   LSTM Yes:       {(df_dict['Use_for_LSTM'] == 'Yes').sum()}")
print(f"   LSTM Optional:  {(df_dict['Use_for_LSTM'] == 'Optional').sum()}")

print("\n📋 Sample dictionary entries:")
print(df_dict.head(10))
print("-"*80)

Step 29: Generating Feature Dictionary
--------------------------------------------------------------------------------
⚠️  Features in DF but not in dictionary: {'Value_Traded_Cr', 'Daily_Return'}

✅ Saved feature_dictionary.csv
   Total features documented: 176

📊 By Category:
Category
Risk-Adjusted      16
Drawdown           12
HH/LL              12
Technical          12
Date               11
Cross-Sectional    11
Volume             11
Sector-Relative     9
Index-Relative      9
Returns             9
Distribution        8
Moving Average      7
Base                7
Beta/Risk           6
Market Regime       6
Volatility          6
Up-Days             6
Market Cap          5
52-Week             4
Price Action        4
Sector              3
Unknown             2
Name: count, dtype: int64

📊 LSTM/XGB Usage:
   XGBoost Yes:    164
   LSTM Yes:       48
   LSTM Optional:  80

📋 Sample dictionary entries:
            Feature_Name Category                                 Description  \
0   

## Step 30: Generate Feature Health Report

In [34]:
print("Step 30: Generating Feature Health Report")
print("-"*80)

# Only numeric columns for stats
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()

health = []
for col in df.columns:
    entry = {
        'Feature_Name': col,
        'Dtype': str(df[col].dtype),
        'NaN_Count': int(df[col].isna().sum()),
        'NaN_Pct': round(df[col].isna().sum() / len(df) * 100, 2),
        'Unique_Values': int(df[col].nunique()),
    }

    if col in numeric_features:
        non_na = df[col].dropna()
        if len(non_na) > 0:
            entry.update({
                'Min': float(non_na.min()),
                'Max': float(non_na.max()),
                'Mean': float(non_na.mean()),
                'Std': float(non_na.std()),
                'Median': float(non_na.median()),
            })
        else:
            entry.update({'Min': None, 'Max': None, 'Mean': None, 'Std': None, 'Median': None})
    else:
        entry.update({'Min': None, 'Max': None, 'Mean': None, 'Std': None, 'Median': None})

    health.append(entry)

df_health = pd.DataFrame(health)
df_health.to_csv(OUTPUT_FILE_HEALTH, index=False)

print(f"✅ Saved {OUTPUT_FILE_HEALTH}")
print(f"   Features analyzed: {len(df_health)}")

print("\n📊 Feature health summary:")
print(df_health[['Feature_Name', 'NaN_Pct', 'Mean', 'Std']].head(15))
print("-"*80)

Step 30: Generating Feature Health Report
--------------------------------------------------------------------------------
✅ Saved feature_health_report.csv
   Features analyzed: 175

📊 Feature health summary:
       Feature_Name  NaN_Pct          Mean           Std
0              Date     0.00           NaN           NaN
1              Open     0.00  7.539123e+02  3.745060e+03
2              High     0.00  7.660266e+02  3.791451e+03
3               Low     0.00  7.432625e+02  3.704251e+03
4             Close     0.00  7.549987e+02  3.750717e+03
5            Volume     0.00  3.445840e+06  1.739438e+07
6            Ticker     0.00           NaN           NaN
7      Value_Traded     0.00  5.533454e+08  1.939328e+09
8   Value_Traded_Cr     0.00  5.533455e+01  1.942300e+02
9      Daily_Return     0.02  1.012956e-03  3.287593e-02
10        Return_1d     0.02  1.016102e-01  3.300713e+00
11        Return_2d     0.05  1.932027e-01  4.463294e+00
12        Return_3d     0.07  2.854655e-01  5.395

## Step 30b: First-Valid-Date Report (FIX V5)

For each feature, report the row index of its first non-NaN value **within each stock**, averaged across stocks. This makes window-start bugs obvious: e.g. `MaxDrawdown_252d` should first appear ~252 rows into each stock, NOT ~504. Eyeball this table to confirm every feature starts at the expected horizon.

In [35]:
print("Step 30b: First-valid-date report")
print("-"*80)

# For each ticker, find the position (0-indexed) of first non-NaN per column,
# then average across tickers. Compare against the feature's expected horizon.
def first_valid_pos(s):
    pos = s.reset_index(drop=True).first_valid_index()
    return pos if pos is not None else np.nan

# Sample up to 50 tickers with long history for speed
ticker_counts = df.groupby('Ticker').size().sort_values(ascending=False)
sample_tickers = ticker_counts.head(50).index.tolist()
df_sample = df[df['Ticker'].isin(sample_tickers)]

feature_cols = [c for c in df.columns if c not in ['Date', 'Ticker', 'Sector', 'Industry', 'Index_Classification']]

rows = []
for col in feature_cols:
    avg_first = df_sample.groupby('Ticker')[col].apply(first_valid_pos).mean()
    rows.append({'Feature': col, 'Avg_First_Valid_Row': round(avg_first, 1) if pd.notna(avg_first) else None})

df_firstvalid = pd.DataFrame(rows).sort_values('Avg_First_Valid_Row', ascending=False, na_position='last')

# Save and show the worst (latest-starting) features
df_firstvalid.to_csv('feature_first_valid_report.csv', index=False)
print("\u2705 Saved feature_first_valid_report.csv")
print("\nFeatures that start LATEST (check these for window-start bugs):")
print(df_firstvalid.head(25).to_string(index=False))
print("\n\ud83d\udca1 Expected: 252d-horizon features ~252, 120d ~120, 60d ~60, etc.")
print("   If something is ~2x its horizon, suspect a double-window bug.")
print("-"*80)

Step 30b: First-valid-date report
--------------------------------------------------------------------------------


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 104, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 1307-1308: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/zmq/eventloop/zmqstream.py", line 551, in _run_callback
    f = callback(*args, **kwargs)
        ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py", line 120, in _handle_event
    event_f()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py", line 518, in _flush
    self.session.send(
  File "/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py", line 848, in send
    

## Step 31: Final Summary

In [36]:
total_time = time.time() - overall_start

print("="*80)
print("✅ CODE 3 V4 — COMPLETE")
print("="*80)
print(f"""
📊 Final Dataset:
   • Rows: {len(df):,}
   • Features: {len(df.columns)}
   • Unique stocks: {df['Ticker'].nunique()}
   • Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}

📁 Output Files:
   1. {OUTPUT_FILE_FEATURES} — Main dataset
   2. {OUTPUT_FILE_DICT} — Feature dictionary (reference for modeling)
   3. {OUTPUT_FILE_HEALTH} — Feature health report
   4. feature_first_valid_report.csv — Window-start sanity check

⏱️  Total runtime: {total_time/60:.1f} minutes

📚 How to use Feature Dictionary in downstream codes:

   import pandas as pd
   df_dict = pd.read_csv('feature_dictionary.csv')

   # For XGBoost
   xgb_features = df_dict[df_dict['Use_for_XGBoost'] == 'Yes']['Feature_Name'].tolist()

   # For LSTM (lean)
   lstm_features = df_dict[df_dict['Use_for_LSTM'] == 'Yes']['Feature_Name'].tolist()

   # Load main dataset
   df = pd.read_csv('data_features.csv')
   df_xgb = df[xgb_features + ['Date', 'Ticker']]  # Add identifiers
   df_lstm = df[lstm_features + ['Date', 'Ticker']]

➡️  Next Step: Run Code 4 (clustering) — cluster features will be added separately later
""")
print("="*80)

✅ CODE 3 V4 — COMPLETE

📊 Final Dataset:
   • Rows: 2,417,660
   • Features: 175
   • Unique stocks: 570
   • Date range: 2007-01-02 to 2026-06-12

📁 Output Files:
   1. data_features.parquet — Main dataset
   2. feature_dictionary.csv — Feature dictionary (reference for modeling)
   3. feature_health_report.csv — Feature health report
   4. feature_first_valid_report.csv — Window-start sanity check

⏱️  Total runtime: 8.7 minutes

📚 How to use Feature Dictionary in downstream codes:

   import pandas as pd
   df_dict = pd.read_csv('feature_dictionary.csv')

   # For XGBoost
   xgb_features = df_dict[df_dict['Use_for_XGBoost'] == 'Yes']['Feature_Name'].tolist()

   # For LSTM (lean)
   lstm_features = df_dict[df_dict['Use_for_LSTM'] == 'Yes']['Feature_Name'].tolist()

   # Load main dataset
   df = pd.read_csv('data_features.csv')
   df_xgb = df[xgb_features + ['Date', 'Ticker']]  # Add identifiers
   df_lstm = df[lstm_features + ['Date', 'Ticker']]

➡️  Next Step: Run Code 4 (clusteri

## Step 32: Download Output Files (Optional)

In [37]:
# Uncomment to download files in Colab
# from google.colab import files
# print("Downloading files...")
# files.download(OUTPUT_FILE_DICT)        # Feature dictionary (small, quick)
# files.download(OUTPUT_FILE_HEALTH)      # Feature health (small, quick)
# files.download(OUTPUT_FILE_FEATURES)    # Main dataset (~1.5 GB, slow)
# print("✅ Done")